In [11]:
import os
from dotenv import load_dotenv
import requests
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
import asyncio

In [7]:
agent = Agent(name = "Sarcasm", instructions="Always make fun of user input", model = "gpt-5.4-mini")

In [8]:
result = await Runner.run(agent, "Sunflower is my favorite flower")

In [ ]:
print(result.final_output)

### Difference between sequential and parallel Runs

In [ ]:
import time

start = time.time()
await Runner.run(agent, "Sunflower is my favorite flower")
await Runner.run(agent, "I love eating pizza")
print("sequential:", time.time() - start)

start = time.time()
await asyncio.gather(
    Runner.run(agent, "Sunflower is my favorite flower"),
    Runner.run(agent, "I love eating pizza"),
)
print("parallel:", time.time() - start)

### What you pass to Runner.run

### await Runner.run(
    agent,                # required: which agent to start with /n
    "user prompt",        # required: a string, or a list of message dicts
    session=session,      # optional: memory
    max_turns=10,         # optional: loop limit
    context=my_obj,       # optional: your own object, passed to tools
    run_config=...,       # optional: run-level settings
)

### Where tools and handoffs live

### agent = Agent(
    name="Color Teller",
    instructions="...",
    model="gpt-5.4-mini",
    tools=[],        # optional, default empty
    handoffs=[],     # optional, default empty
)

### What happens depends on what you configured:

### Agent config	                What the loop does
### No tools, no handoffs (yours)	    One LLM call, final answer, done
### tools=[push_tool]	                 LLM may reply with a tool call. The Runner runs the function, adds the result to the messages, and calls the LLM again
### handoffs=[other_agent]	LLM may reply with a handoff. The Runner switches to the other agent and continues

Having a tool configured does not force a call. The LLM chooses based on your instructions, the tool description, and the user prompt. The Runner only reacts to what the LLM returns.

### await Runner.run(agent, "Sunflower is my favorite flower")        # string

### await Runner.run(agent, [                                         # list of messages
    {"role": "user", "content": "Hi, my name is Ed"},
    {"role": "assistant", "content": "Hi Ed!"},
    {"role": "user", "content": "What's my name?"},
])

In [ ]:
result.to_input_list()

In [15]:
### Without trace: 2 separate traces, both named "Agent workflow"
await Runner.run(agent, "I like taking shower at night")
await Runner.run(agent, "I like vampire diaries")

# With trace: 1 trace named "Joke session", containing both runs
with trace("Joke session"):
    await Runner.run(agent, "I like taking shower at night")
    await Runner.run(agent, "I like vampire diaries")

In [16]:
with trace("Joke sessions"):
    await Runner.run(agent, "I dont like my neighbours because they are too noisy")
    await Runner.run(agent, "Maybe I love them")

In [ ]:
result = Runner.run_streamed(agent, input="...")
# 1. Starts the agent loop in the background and returns right away.
#    The background task calls the LLM and puts each incoming event into a queue.

async for event in result.stream_events():
    # 2. Each iteration takes the next event out of that queue
    #    (it waits if nothing has arrived yet).

    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        # 3. Filter: only keep events that are raw LLM output AND a text chunk.
        #    Every other event is skipped.

        print(event.data.delta, end="", flush=True)
        # 4. Print that chunk right now, with no newline, so the text builds up on screen.

# 5. When the background task finishes, it puts a "done" marker in the queue,
#    the loop ends, and result.final_output now holds the full text.

In [ ]:
result = Runner.run_streamed(agent, input="Describe a sunflower field in 5 short sentences")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

#### .delta is the small text chunk, often a word or a fragment.
#### end="" stops print from adding a newline after every chunk.
#### flush=True pushes the text to the screen immediately instead of letting Python buffer it.

### After the loop finishes, result.final_output holds the complete text.


## Adding a tool

In [ ]:
load_dotenv()

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")


In [38]:
def push(message):
    print(f"push : {message}")
    payload = {"user" : pushover_user, "token": pushover_token, "message" : message}
    requests.post(pushover_url, data = payload)

In [ ]:
push("Hey Kavya!! Its Kavya")

In [ ]:
push

In [48]:
@function_tool
def push_tool(message:str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user" : pushover_user, "token": pushover_token, "message" : message}
    result = requests.post(pushover_url, data = payload)
    return f"Push sent with API status code {result}"

In [ ]:
push_tool

In [ ]:
push_tool.description

In [50]:
notifier = Agent(name="Notifier", instructions="You notify the user upon motivation request", model="gpt-5.4-mini", tools=[push_tool])

In [ ]:
with trace("Motivate Myself!!"):
    result = await Runner.run(notifier, "Notify user that She is doing good, Keep up with her consistency and efforts in learning new things to whomever you send this message to;")

print(result.final_output)

### Memory approach - 1

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Kavya.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]

In [ ]:
response = await Runner.run(agent,next_input)
print(response.final_output)

### Memory approach - 2

In [ ]:
session = SQLiteSession("Test session")


In [62]:
response = await Runner.run(agent, "Hi there my name is Kavya", session=session)

In [ ]:
response.final_output

In [64]:
response = await Runner.run(agent, "whats my name??", session=session)

In [ ]:
response.final_output